# Inference speed: HF Transformers vs vLLM

Benchmark single-prompt latency and throughput on the same
Llama-3.2-1B-Instruct checkpoint across:

1. HF Transformers (eager generation)
2. vLLM with `gpu_memory_utilization=0.2`
3. vLLM with `gpu_memory_utilization=0.7`

Decoding is stochastic (`do_sample=True` / `temperature > 0`) on
both backends, with matching `temperature` and `top_p`. Each timed
iteration uses a fresh seed `base_seed + i` — outputs differ from
run to run but are reproducible across re-executions of the cell.
Token counts vary per iteration since the sampled completions hit
EOS at different points; throughput (tokens / second) is the
robust metric to compare.

Each backend includes one untimed warmup pass to exclude cudagraph
capture / JIT compilation from the latency.

Note: `gpu_memory_utilization` mainly affects vLLM's KV-cache pool
size, which matters for *concurrent* requests. On a single prompt
the two vLLM settings should land within noise of each other.

## Setup

In [1]:
import os
os.environ["VLLM_CONFIGURE_LOGGING"] = "0"
import logging
logging.basicConfig(format='%(message)s', level=logging.FATAL+1)

import gc
import time

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from vllm import LLM, SamplingParams

In [2]:
# Dataset and model paths
base_dir = '/groups/chichengz/tnn/datasets/'

dataset_dir = base_dir + "/prm800k/math_splits"

# Causal LM under test (swap to Llama-3.2-1B-Instruct to compare)
# llm_dir = base_dir + "/Llama-3.2-1B-Instruct"
llm_dir = base_dir + "Qwen2.5-3B-Instruct"

In [3]:
# Benchmark prompt + decoding config
prompt = (
    r'If $f(x) = \frac{3x-2}{x-2}$, what is the value of '
    r'$f(-2) + f(-1) + f(0)$? Express your answer as a common fraction.'
)
max_new_tokens = 1024
num_runs = 10

# Stochastic decoding — same params on both backends for a fair comparison
temperature = 0.8
top_p = 0.95
base_seed = 123    # iteration i uses seed = base_seed + i

In [4]:
def gpu_mem_used_gb(device=0):
    """Driver-level used GPU memory; sees both PyTorch and vLLM allocs."""
    free, total = torch.cuda.mem_get_info(device)
    return (total - free) / (1024**3)


def measure_inference(
    backend, model, tokenizer, prompt, max_new_tokens, num_runs,
    temperature, top_p, base_seed=123, warmup=1,
):
    """Time `num_runs` stochastic generations and report stats.

    `backend` is "hf" or "vllm". A `warmup` untimed pass absorbs
    cudagraph capture / JIT overhead. Each timed iteration uses
    `base_seed + i` so runs differ but are reproducible across
    re-executions. Only newly generated tokens are counted.
    """
    def _generate(seed):
        if backend == "vllm":
            params = SamplingParams(
                temperature=temperature,
                top_p=top_p,
                max_tokens=max_new_tokens,
                seed=seed,
            )
            out = model.generate(prompt, params, use_tqdm=False)
            tok_ids = out[0].outputs[0].token_ids
            return out[0].outputs[0].text, len(tok_ids)
        else:
            inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
            prompt_len = inputs['input_ids'].shape[1]
            pad_id = tokenizer.pad_token_id or tokenizer.eos_token_id
            torch.manual_seed(seed)        # seeds CPU + CUDA RNGs
            with torch.no_grad():
                out_ids = model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    do_sample=True,
                    temperature=temperature,
                    top_p=top_p,
                    pad_token_id=pad_id,
                )
            new_ids = out_ids[0, prompt_len:]
            return tokenizer.decode(new_ids, skip_special_tokens=True), len(new_ids)

    # Warmup: use a seed outside the timed range so timed runs stay reproducible
    for w in range(warmup):
        _generate(base_seed + 10_000 + w)

    total_time = 0.0
    total_tokens = 0
    text = ""
    for i in range(num_runs):
        start = time.perf_counter()
        text, n_tokens = _generate(base_seed + i)
        total_time += time.perf_counter() - start
        total_tokens += n_tokens

    latency = total_time / num_runs
    throughput = total_tokens / total_time
    avg_tokens = total_tokens / num_runs
    return latency, throughput, avg_tokens, text

## HF Transformers (baseline)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(llm_dir)
model_hf = AutoModelForCausalLM.from_pretrained(
    llm_dir,
    dtype="float16",
    device_map="cuda:0",
)
model_hf.eval()

print(f'#--- GPU memory used: {gpu_mem_used_gb():.2f} GB')
print(model_hf.dtype)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

#--- GPU memory used: 6.12 GB
torch.float16


In [6]:
latency_hf, throughput_hf, avg_tokens_hf, text_hf = measure_inference(
    "hf", model_hf, tokenizer, prompt, max_new_tokens, num_runs,
    temperature=temperature, top_p=top_p, base_seed=base_seed,
)
print(
    f"HF Transformers   - latency: {latency_hf:.4f}s, "
    f"throughput: {throughput_hf:.2f} tok/s, "
    f"avg tokens: {avg_tokens_hf:.1f}"
)

HF Transformers   - latency: 17.5157s, throughput: 26.96 tok/s, avg tokens: 472.3


In [7]:
# Free HF before loading vLLM so they don't fight over the GPU
del model_hf
del tokenizer
gc.collect()
torch.cuda.empty_cache()
print(f'#--- GPU memory used: {gpu_mem_used_gb():.2f} GB')

#--- GPU memory used: 6.12 GB


## vLLM with `gpu_memory_utilization=0.5`

Small KV-cache pool — enough headroom for one prompt but not for
concurrent batching at long contexts.

In [8]:
llm_vllm = LLM(
    model=llm_dir,
    tensor_parallel_size=1,
    gpu_memory_utilization=0.5,
    max_model_len=5000,
    dtype="float16",
    seed=123,
)

print(f'#--- GPU memory used: {gpu_mem_used_gb():.2f} GB')

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
Cannot use FA version 2 is not supported due to FA2 is only supported on devices with compute capability >= 8
<frozen importlib._bootstrap_external>:1241: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
<frozen importlib._bootstrap_external>:1241: FutureWarning: The cuda.nvrtc module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.nvrtc module instead.
Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  50% Completed | 1/2 [00:02<00:02,  2.08s/it]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:03<00:00,  1.58s/it]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:03<00:00,  1.66s/it]

Capturing CUDA graphs (mixed pref

#--- GPU memory used: 22.96 GB


In [9]:
latency_v02, throughput_v02, avg_tokens_v02, text_v02 = measure_inference(
    "vllm", llm_vllm, None, prompt, max_new_tokens, num_runs,
    temperature=temperature, top_p=top_p, base_seed=base_seed,
)
print(
    f"vLLM gpu_mem=0.2  - latency: {latency_v02:.4f}s, "
    f"throughput: {throughput_v02:.2f} tok/s, "
    f"avg tokens: {avg_tokens_v02:.1f}"
)

vLLM gpu_mem=0.2  - latency: 4.2631s, throughput: 113.11 tok/s, avg tokens: 482.2


In [ ]:
# Free the first vLLM engine before reloading at a higher pool size
del llm_vllm
gc.collect()
torch.cuda.empty_cache()
print(f'#--- GPU memory used: {gpu_mem_used_gb():.2f} GB')

#--- GPU memory used: 6.12 GB


## vLLM with `gpu_memory_utilization=0.9`

Large KV-cache pool. For a single prompt this should match the
0.2 setting within noise; the difference shows up under concurrent
batching (more sequences live in cache at once).

In [11]:
llm_vllm = LLM(
    model=llm_dir,
    tensor_parallel_size=1,
    gpu_memory_utilization=0.8,
    max_model_len=5000,
    dtype="float16",
    seed=123,
)

print(f'#--- GPU memory used: {gpu_mem_used_gb():.2f} GB')

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
EngineCore failed to start.
Traceback (most recent call last):
  File "/home/u20/tnguyen9210/micromamba/envs/py311/lib/python3.11/site-packages/vllm/v1/engine/core.py", line 1082, in run_engine_core
    engine_core = EngineCoreProc(*args, engine_index=dp_rank, **kwargs)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/u20/tnguyen9210/micromamba/envs/py311/lib/python3.11/site-packages/vllm/tracing/otel.py", line 178, in sync_wrapper
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/home/u20/tnguyen9210/micromamba/envs/py311/lib/python3.11/site-packages/vllm/v1/engine/core.py", line 848, in __init__
    super().__init__(
  File "/home/u20/tnguyen9210/micromamba/envs/py311/lib/python3.11/site-packages/vllm/v1/engine/core.py", line 114, in __init__
    self.model_executor = executor_class(vllm_config)
    

RuntimeError: Engine core initialization failed. See root cause above. Failed core proc(s): {}

In [ ]:
latency_v07, throughput_v07, avg_tokens_v07, text_v07 = measure_inference(
    "vllm", llm_vllm, None, prompt, max_new_tokens, num_runs,
    temperature=temperature, top_p=top_p, base_seed=base_seed,
)
print(
    f"vLLM gpu_mem=0.7  - latency: {latency_v07:.4f}s, "
    f"throughput: {throughput_v07:.2f} tok/s, "
    f"avg tokens: {avg_tokens_v07:.1f}"
)

vLLM gpu_mem=0.7  - latency: 1.9540s, throughput: 251.07 tok/s, avg tokens: 490.6


## Summary

In [ ]:
header = f"{'Backend':<22}{'Latency (s)':>14}{'Tok/s':>12}{'Avg tok':>12}"
print(header)
print('-' * len(header))
rows = [
    ('HF Transformers',  latency_hf,  throughput_hf,  avg_tokens_hf),
    ('vLLM gpu_mem=0.2', latency_v02, throughput_v02, avg_tokens_v02),
    ('vLLM gpu_mem=0.7', latency_v07, throughput_v07, avg_tokens_v07),
]
for name, lat, tput, ntok in rows:
    print(f"{name:<22}{lat:>14.4f}{tput:>12.2f}{ntok:>12.1f}")

print(f"\nSample completion (HF, last run):\n{text_hf[:400]}...")

Backend                  Latency (s)       Tok/s     Avg tok
------------------------------------------------------------
HF Transformers               8.6875       59.95       520.8
vLLM gpu_mem=0.2              1.9547      250.98       490.6
vLLM gpu_mem=0.7              1.9540      251.07       490.6

Sample completion (HF, last run):
 To evaluate $f(-2)$, plug in $x=-2$ into $f(x)$. Similarly, to evaluate $f(-1)$, plug in $x=-1$ into $f(x)$. To evaluate $f(0)$, plug in $x=0$ into $f(x)$. Evaluate each of these expressions. \begin{align*} f(-2) &amp;= \frac{3(-2)-2}{(-2)-2}\\ &= \frac{-8-2}{-4}\\ &= \frac{-10}{-4}\\ &= \frac{5}{2} \end{align*} \begin{align*} f(-1) &amp;= \frac{3(-1)-2}{(-1)-2}\\ &= \frac{-3-2}{-3}\\ &= \frac{-5...
